# 7.2 - Streaming & Real-Time Patterns

**Module 7 - LLM API Engineering** | Netsetos GenAI Engineering

This notebook builds the token-streaming path end to end: async SDK streaming from Claude, OpenAI and Gemini; a FastAPI Server-Sent-Events endpoint; the browser code that consumes it; a WebSocket bridge that supports mid-generation interruption; a Cloud Run deploy; a two-provider race; and a real-time speech-to-speech sketch. Along the way it shows exactly where a proxy silently un-streams you and the two lines that fix it.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough. Most cells need at least one provider key; the server and deploy cells are meant to run in a shell, not inline.

## Setup

Install the client libraries this lesson streams through. Everything else is standard library (`asyncio`, `time`, `json`), so this one `pip` line is the whole dependency list.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install "anthropic>=0.40.0" openai google-genai python-dotenv websockets -q  # noqa


**Explanation**

A single commented install line - uncomment it on Colab or a fresh environment. Note `google-genai` (not the retired `google-generativeai`) is what gives Gemini an async client, which the whole lesson relies on.

**How the code works - step by step**
- **`anthropic>=0.40.0`** - the Claude SDK, with the async `messages.stream()` context manager.
- **`openai`** - the OpenAI SDK, used here via `AsyncOpenAI` with `stream=True`.
- **`google-genai`** - the current Gemini SDK; its `.aio` surface is the async client the old SDK lacked.
- **`python-dotenv`** - loads keys from a `.env` file so nothing is hardcoded.
- **`websockets`** - the raw WebSocket client used in the final real-time-audio cell.

**In one line:** three provider SDKs plus dotenv and a WebSocket client.

**What you'll see:** (no output - environment prep)

## API keys

Load provider keys from the environment instead of hardcoding them. Any one key is enough to follow along; the multi-provider demos are richer with two or more.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Pure environment prep, not a model call. `setdefault` only fills a variable if it is not already set, so real keys already exported in your shell are left untouched and the empty-string fallbacks just keep imports from crashing.

**How the code works - step by step**
- **`ANTHROPIC_API_KEY`** - from console.anthropic.com, read by `AsyncAnthropic`.
- **`OPENAI_API_KEY`** - from platform.openai.com, read by `AsyncOpenAI`.
- **`GOOGLE_API_KEY`** - from aistudio.google.com, read by `genai.Client()`.
- **`os.environ.setdefault(...)`** - sets each only if absent, so it never clobbers a real key you already exported.

**In one line:** make sure the three keys exist in the environment without ever writing them into the code.

**What you'll see:** (no output - environment prep)

## The cold open - the stream that arrived as a parcel

Same endpoint, same code, two very different afternoons: at 9 AM on your laptop a token lands every ~40ms; at 4 PM behind staging's nginx the client sees 28 seconds of nothing, then all 412 tokens in one blob. This cell is the whole lesson's motivating scene.

In [ ]:
# Output: 9:00 AM - your laptop:
# $ curl -N localhost:8000/chat/stream -d '{"prompt":"..."}'
#   data: {"token": "The"}        <- 210ms
#   data: {"token": " capital"}   <- 251ms   ...a new token every ~40ms. Perfect.
#
# 4:00 PM - staging, behind nginx, in front of your stakeholders:
# $ curl -N https://staging.yourapp.com/chat/stream -d '{"prompt":"..."}'
#
#   [ ............ 28 seconds of nothing ............ ]
#
#   data: {"token": "The"} data: {"token": " capital"} ... ALL 412 tokens, one blob.
#
# Your PM: "I thought you said it streams?"
# Your code did not change. The proxy between you and the user did.

**Explanation**

A comment-only narrative cell - there is no runnable Python here, it is a transcript of two `curl` sessions. The point it plants: your code did not change, the proxy between you and the user did, and that is the failure Steps 5 and 7 mechanize and fix.

**How the code works - step by step**
- **9:00 AM block** - `curl -N localhost:8000` shows frames arriving individually (`The` at 210ms, ` capital` at 251ms), the streaming success case.
- **4:00 PM block** - the identical request through `https://staging...` behind nginx buffers everything, then flushes all tokens as a single blob.
- **The takeaway line** - the code is unchanged; a reverse proxy in the middle silently re-batched the stream.

**In one line:** streaming is an agreement across every hop, and any hop can quietly break it.

**What you'll see:** No program runs - the cell is a commented-out record of two curl sessions contrasting a smooth per-token stream against a proxy that dumps all 412 tokens at once after a 28-second stall.

## 1 - Stream from all three providers

Stream tokens from Claude, OpenAI and Gemini on their async clients, and time each one's TTFT (time to first token) with a single shared stopwatch. This is the primitive every later cell reuses.

> **Needs at least one provider key** (ANTHROPIC_API_KEY, OPENAI_API_KEY, and/or GOOGLE_API_KEY).

In [ ]:
# SDK STREAMING - all three providers on ASYNC clients
# pip install "anthropic>=0.40.0" openai google-genai
# (google-generativeai reached end-of-life 2025-11-30 - google-genai replaces it)
import asyncio, os, time

prompt = "Explain streaming in LLM APIs in 3 sentences."

async def timed(name, agen):
    # one timing harness for all three providers
    start = time.time(); ttft = None
    async for text in agen:
        if ttft is None: ttft = time.time() - start  # only the FIRST chunk sets it
        print(text, end="", flush=True)
    print(f"\n{name} TTFT: {ttft*1000:.0f}ms\n")

async def main():
    # Claude - stream() context manager, timeout + typed errors built in
    import anthropic
    cl = anthropic.AsyncAnthropic(timeout=30.0)
    async def claude():
        async with cl.messages.stream(model="claude-sonnet-4-6", max_tokens=200,
            messages=[{"role":"user","content":prompt}]) as st:
            async for t in st.text_stream: yield t
    await timed("Claude", claude())

    # OpenAI
    from openai import AsyncOpenAI
    oai = AsyncOpenAI(timeout=30.0)
    async def gpt():
        stream = await oai.chat.completions.create(model="gpt-5.4",
            messages=[{"role":"user","content":prompt}], stream=True)
        async for chunk in stream:
            if chunk.choices[0].delta.content: yield chunk.choices[0].delta.content
    await timed("OpenAI", gpt())

    # Gemini - the google-genai async client
    from google import genai
    g = genai.Client()  # reads GOOGLE_API_KEY
    async def gemini():
        stream = await g.aio.models.generate_content_stream(
            model="gemini-3.5-flash", contents=prompt)
        async for chunk in stream:
            if chunk.text: yield chunk.text
    await timed("Gemini", gemini())

asyncio.run(main())

# Output: (representative run; medians move - see the sourced table in Step 5)
# Streaming sends each token the moment the model produces it, so users read...
# Claude TTFT: 487ms
# Streaming lets an API return partial output while generation continues...
# OpenAI TTFT: 1096ms
# Streaming delivers tokens incrementally over one HTTP response...
# Gemini TTFT: 342ms
# Note: total generation time is unchanged - only the FIRST paint moved.

**Explanation**

Three provider streams driven through one timing harness - the differences live in the plumbing, not the concept. `timed()` is the reusable belt: it starts a clock on request and stamps TTFT the instant the first chunk lands, printing each token as it arrives. All three clients get `timeout=30.0`, a real network timeout that cancels a hung connection.

**How the code works - step by step**
- **`timed(name, agen)`** - one stopwatch for all three belts; because only `ttft is None` sets the value, TTFT is captured on the FIRST chunk and never overwritten, then every token is printed with `flush=True`.
- **`claude()`** - `AsyncAnthropic().messages.stream(...)` is a context manager; `.text_stream` yields plain text deltas and the `async with` block cleans up the connection even on early exit.
- **`gpt()`** - `AsyncOpenAI(...).chat.completions.create(stream=True)` yields chunks whose `delta.content` is `None` on bookkeeping frames (role header, finish frame) - the `if` guard skips those.
- **`gemini()`** - `genai.Client().aio.models.generate_content_stream(...)` is the async surface of google-genai; the old SDK had no async client, which is why streaming it used to block the event loop.

**In one line:** three belts, one stopwatch - learn the streaming call once, reuse it everywhere.

**What you'll see:** Each provider's 3-sentence answer prints token-by-token, followed by a TTFT line - representatively `Claude TTFT: 487ms`, `OpenAI TTFT: 1096ms`, `Gemini TTFT: 342ms`. Total generation time is unchanged; only the first paint moved earlier.

## 2 - A FastAPI SSE endpoint

Wrap the streaming call in an HTTP endpoint that pushes tokens to the browser as Server-Sent Events. This is the server half of a streaming chat: named frames, a disconnect guard, and the exact headers that stop a proxy from buffering.

> **Run in a shell, not inline:** `uvicorn main:app --reload`, then `curl -N -X POST localhost:8000/chat/stream -d '{"prompt":"hi"}'`.

In [ ]:
# FASTAPI SSE - stream LLM tokens to the browser
# run: uvicorn main:app --reload     test: curl -N -X POST localhost:8000/chat/stream -d '{"prompt":"hi"}'
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from google import genai
import json

app = FastAPI()
# scope CORS to YOUR frontend - a wildcard lets any website spend your tokens
app.add_middleware(CORSMiddleware, allow_origins=["https://yourapp.example"],
                   allow_methods=["POST"], allow_headers=["content-type"])
client = genai.Client()  # reads GOOGLE_API_KEY from the environment

def sse(event: str, data: dict) -> str:
    # the SSE wire format: an event name, a data line, and a BLANK line as the frame boundary
    return f"event: {event}\ndata: {json.dumps(data)}\n\n"

@app.post("/chat/stream")
async def chat_stream(request: Request):
    body = await request.json()
    async def gen():
        tokens = 0
        try:
            yield sse("status", {"state": "generating"})
            stream = await client.aio.models.generate_content_stream(
                model="gemini-3.5-flash", contents=body["prompt"])
            async for chunk in stream:
                if await request.is_disconnected():   # tab closed? stop generating, stop paying
                    print(f"client gone after {tokens} tokens"); return
                if chunk.text:
                    tokens += 1
                    yield sse("token", {"text": chunk.text})
            yield sse("done", {"tokens": tokens})
        except Exception as e:
            yield sse("error", {"message": str(e)})  # never die silently mid-stream
    return StreamingResponse(gen(), media_type="text/event-stream",
        headers={"Cache-Control":"no-cache", "X-Accel-Buffering":"no"})
# X-Accel-Buffering: no tells nginx / Cloud Run: do NOT buffer this response

# Output: (curl -N, representative)
# event: status
# data: {"state": "generating"}
# event: token
# data: {"text": "Streaming"}
# ...a new frame every ~40ms...
# event: done
# data: {"tokens": 42}

**Explanation**

An async FastAPI route that returns a `StreamingResponse` of `text/event-stream`. The core idea is that SSE is just a text wire format - an event name, a data line, and a blank line as the frame boundary - and that every await point keeps the single event loop free for other requests. `X-Accel-Buffering: no` is the one header that tells nginx/Cloud Run not to re-batch this response.

**How the code works - step by step**
- **`sse(event, data)`** - builds the wire format: `event:` names the frame so the client can route status vs token vs error, `data:` carries JSON, and the trailing blank line is the boundary; drop it and the browser waits forever.
- **`CORSMiddleware(allow_origins=[...])`** - scoped to your own frontend, because a wildcard lets any website spend your tokens.
- **`gen()` + the async genai client** - every wait is an `await`, so other requests keep flowing while this one streams; a sync SDK here would freeze the whole loop per chunk.
- **`request.is_disconnected()`** - the billing guard: if the tab closes, generation stops instead of producing (and charging) into a void.
- **the `except` arm** - a mid-stream provider error becomes an `event: error` frame the client can render, rather than a silently truncated answer.
- **response headers** - `Cache-Control: no-cache` and `X-Accel-Buffering: no` tell proxies to pass frames through unbuffered.

**In one line:** name your frames, await your waits, and always plan for the closed tab.

**What you'll see:** Under `curl -N` you see the frames arrive one at a time: an `event: status` frame, then `event: token` frames roughly every 40ms, and finally an `event: done` frame carrying the token count.

## 3 - The browser consumer

The client side of Step 2's endpoint: fetch the stream and parse SSE frames by hand. `EventSource` can't be used here because this endpoint needs POST, a JSON body, and headers - so it's `fetch()` plus a `ReadableStream` reader.

In [ ]:
%%javascript
// THE CONSUMER - fetch() + ReadableStream (EventSource cannot POST or send headers)
const controller = new AbortController();
stopBtn.onclick = () => controller.abort();      // the "Stop generating" button

const res = await fetch("/chat/stream", {
  method: "POST",
  headers: {"content-type": "application/json"},
  body: JSON.stringify({prompt}),
  signal: controller.signal,
});

const reader = res.body.pipeThrough(new TextDecoderStream()).getReader();
let buf = "";
while (true) {
  const {value, done} = await reader.read();
  if (done) break;
  buf += value;
  let idx;
  while ((idx = buf.indexOf("\n\n")) !== -1) {   // one SSE frame per blank line
    const frame = buf.slice(0, idx); buf = buf.slice(idx + 2);
    const ev = /^event: (.*)$/m.exec(frame)?.[1] ?? "message";
    const data = JSON.parse(/^data: (.*)$/m.exec(frame)[1]);
    if (ev === "token") output.textContent += data.text;
    if (ev === "error") showError(data.message);
  }
}
// >>> Output: tokens append to the page every ~40ms; Stop aborts the fetch,
// which triggers request.is_disconnected() server-side. Full circle.

**Explanation**

Browser JavaScript (run via the `%%javascript` cell magic) that reads the response body as a stream and splits it into frames on the blank-line boundary. It closes the loop with Step 2: the Stop button aborts the fetch, which trips `request.is_disconnected()` on the server and halts billing.

**How the code works - step by step**
- **`AbortController` + `stopBtn.onclick`** - the client half of "Stop generating"; `controller.abort()` tears down the connection.
- **`fetch('/chat/stream', {method:'POST', ..., signal})`** - POSTs the prompt with the abort signal attached (this is why EventSource won't do).
- **`res.body.pipeThrough(new TextDecoderStream()).getReader()`** - turns the byte stream into decoded text chunks.
- **the `while` + `buf.indexOf('\n\n')` loop** - accumulates text and extracts exactly one SSE frame per blank line; regexes pull the `event:` name and parse the `data:` JSON.
- **event routing** - `token` frames append to the page, `error` frames call `showError`.

**In one line:** read bytes, split on the blank line, route each frame by its event name.

**What you'll see:** Tokens append to the page every ~40ms as frames arrive; pressing Stop aborts the fetch, which triggers the server's disconnect check - client cancel and server stop-billing are the two halves of one feature.

## 4 - WebSocket bridge with real interruption

Upgrade from one-way SSE to full-duplex: the client can send AND receive at the same time, so a user can interrupt mid-generation or fire a follow-up that replaces the current answer. Interruption is what justifies WebSocket's extra complexity.

> **Run in a shell:** `uvicorn main:app --reload`, then connect a WebSocket client to `/ws/chat`.

In [ ]:
# WEBSOCKET - full duplex WITH REAL interruption
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from google import genai
import asyncio, json

app = FastAPI()
client = genai.Client()

async def generate(ws: WebSocket, prompt: str):
    try:
        stream = await client.aio.models.generate_content_stream(
            model="gemini-3.5-flash", contents=prompt)
        async for chunk in stream:
            if chunk.text: await ws.send_json({"type":"token", "text":chunk.text})
        await ws.send_json({"type": "done"})
    except asyncio.CancelledError:
        await ws.send_json({"type": "cancelled"})  # confirm the interrupt to the client
        raise

@app.websocket("/ws/chat")
async def ws_chat(ws: WebSocket):
    await ws.accept()
    task = None
    try:
        while True:
            msg = json.loads(await ws.receive_text())   # ALWAYS listening - generation runs elsewhere
            if msg["type"] == "stop" and task and not task.done():
                task.cancel()                              # <- the actual interruption
            elif msg["type"] == "prompt":
                if task and not task.done(): task.cancel()  # new question replaces the old answer
                task = asyncio.create_task(generate(ws, msg["text"]))
    except WebSocketDisconnect:
        if task and not task.done(): task.cancel()
# The receive loop and the generation task run CONCURRENTLY.
# That concurrency - not the protocol - is what makes interruption possible.

# Output: (client sends {"type":"stop"} mid-generation)
# {"type":"token","text":"Server-sent"} {"type":"token","text":" events"} ...
# {"type":"cancelled"}
# ...and the very next prompt starts a fresh generation task immediately.

**Explanation**

A FastAPI WebSocket handler whose key move is concurrency, not the protocol: generation runs as a background `asyncio` task while a `while True` receive loop keeps listening. Because the two run concurrently, a `stop` message is actually heard mid-answer - a handler that streamed inside the receive loop never could.

**How the code works - step by step**
- **`generate(ws, prompt)`** - streams tokens as `{'type':'token'}` frames; its `except asyncio.CancelledError` arm sends a `cancelled` confirmation and then re-raises (swallowing CancelledError is an asyncio anti-pattern).
- **`asyncio.create_task(generate(...))`** - runs generation in the background so the receive loop keeps listening DURING generation.
- **the `while True` receive loop** - always reading; a `stop` message calls `task.cancel()`, and a new `prompt` cancels the in-flight task before starting a fresh one (cancel-then-replace).
- **`task.cancel()`** - raises `CancelledError` inside the task at its next `await`, which is the actual interruption.
- **`WebSocketDisconnect`** - the closed-tab guard: cancel the orphaned task so it doesn't stream (and bill) into nowhere.

**In one line:** one ear always open, one mouth cancellable - duplex is a concurrency pattern, not just a protocol.

**What you'll see:** Token frames stream back until the client sends `{"type":"stop"}`, at which point a `{"type":"cancelled"}` frame confirms the interrupt; the very next prompt immediately starts a fresh generation task.

## 5 - Deploy to Cloud Run

Ship the streaming API to Cloud Run with the specific flags that keep streaming intact in production - the wrong defaults buffer your whole response or time out long generations. The key stays in Secret Manager, never in the deploy command.

> **Run in a shell** (gcloud CLI, not Colab).

In [ ]:
%%bash
# (deployment/terminal commands - run in a shell, not Colab, unless configured)
# Cloud Run deployment for streaming API
# store the key in Secret Manager ONCE (never in the deploy command or env vars):
gcloud secrets create google-api-key --data-file=- <<< "$GOOGLE_API_KEY"

gcloud run deploy genai-streaming \
  --source . \
  --region asia-south1 \
  --memory 512Mi --cpu 1 \
  --min-instances 0 --max-instances 10 \
  --timeout 600 \
  --no-cpu-throttling \
  --set-secrets GOOGLE_API_KEY=google-api-key:latest
# add auth in front (IAM/ID tokens or an API key check) before real traffic -
# an open LLM endpoint is a token faucet for whoever finds the URL.

# Critical flags:
# --no-cpu-throttling  Keep CPU active while the response streams (NOTE: this switches
#                      billing to instance-time - the "scales to zero" cost story changes)
# --timeout 600        The DEFAULT is 300s - long generations need explicit headroom
# X-Accel-Buffering:no In the FastAPI response headers (Step 2) - prevents buffering
# --set-secrets        Key comes from Secret Manager, not shell history or the console

# Output: (representative)
# Deploying container to Cloud Run service [genai-streaming]...
# Service URL: https://genai-streaming-xxxxx-el.a.run.app
# curl -N -X POST $URL/chat/stream ... -> event frames arrive one by one, unbuffered

**Explanation**

A shell script (via `%%bash`) that stores the API key in Secret Manager once, then deploys the container with the flags streaming actually needs. The takeaway is that streaming success in prod is a deploy-config problem as much as a code one: CPU throttling, the 300s default timeout, and proxy buffering each silently break the belt.

**How the code works - step by step**
- **`gcloud secrets create ... <<< "$GOOGLE_API_KEY"`** - stores the key in Secret Manager one time, never in the deploy command, env vars, or shell history.
- **`--no-cpu-throttling`** - keeps CPU active while the response streams (note: this switches billing to instance-time, so the "scales to zero" cost story changes).
- **`--timeout 600`** - overrides the 300s default so long generations aren't cut off.
- **`--set-secrets GOOGLE_API_KEY=...`** - injects the key from Secret Manager at runtime.
- **`--min-instances 0 --max-instances 10`** - autoscaling bounds; and the comment reminds you to add auth before real traffic, since an open LLM endpoint is a token faucet.

**In one line:** no CPU throttling + a longer timeout + Secret Manager + the no-buffering header = streaming that survives production.

**What you'll see:** Deploy logs print `Deploying container to Cloud Run...` and then a `Service URL: https://genai-streaming-...run.app`; hitting `$URL/chat/stream` with `curl -N` returns event frames one by one, unbuffered.

## 6 - Race two providers, keep the fastest

Start Claude and Gemini streaming the same prompt at once, take whoever produces the first token, and cancel the loser. This trades roughly double the cost for the lowest possible TTFT.

> **Needs both ANTHROPIC_API_KEY and GOOGLE_API_KEY** for a real race.

In [ ]:
# Race two providers, keep whoever streams the FIRST token, cancel the loser
import asyncio

async def first_token(name, agen_factory):
    try:
        async for text in agen_factory():
            return name, text   # first chunk wins
    except asyncio.CancelledError:
        raise
    except Exception as e:
        return name, f"[{name} failed: {e}]"

# Two async-generator FACTORIES - each returns a fresh stream when called.
# They reuse the exact SDK streaming patterns from Step 1.
import anthropic
from google import genai

PROMPT = "Explain streaming in one sentence."

async def claude_stream():
    cl = anthropic.AsyncAnthropic(timeout=30.0)
    async with cl.messages.stream(model="claude-sonnet-4-6", max_tokens=100,
        messages=[{"role": "user", "content": PROMPT}]) as st:
        async for t in st.text_stream:
            yield t

async def gemini_stream():
    stream = await genai.Client().aio.models.generate_content_stream(
        model="gemini-3.5-flash", contents=PROMPT)
    async for chunk in stream:
        if chunk.text:
            yield chunk.text

async def race(claude_factory, gemini_factory):
    tasks = [
        asyncio.create_task(first_token("claude", claude_factory)),
        asyncio.create_task(first_token("gemini", gemini_factory)),
    ]
    done, pending = await asyncio.wait(
        tasks, timeout=30.0, return_when=asyncio.FIRST_COMPLETED)
    for t in pending:
        t.cancel()   # stop the slower stream - you still pay for it
    return next(iter(done)).result()

winner, text = asyncio.run(race(claude_stream, gemini_stream))
print(winner, text)
# Output: (representative - needs live API keys; which provider's first token
#          arrives first varies by network and load, e.g.)
# gemini Streaming delivers tokens incrementally over one HTTP response...

**Explanation**

A small racing harness built directly on Step 1's streaming patterns. Each provider is wrapped as an async-generator FACTORY (a function that returns a fresh stream when called), and `asyncio.wait(..., return_when=FIRST_COMPLETED)` returns as soon as one side yields its first token; the pending task is cancelled - though you still pay for whatever it generated.

**How the code works - step by step**
- **`first_token(name, agen_factory)`** - runs a stream and returns on the very first chunk; it re-raises `CancelledError` but catches other exceptions so one provider's failure doesn't kill the race.
- **`claude_stream()` / `gemini_stream()`** - async-generator factories reusing the exact SDK streaming calls from Step 1, each returning a fresh stream per call.
- **`race(...)`** - wraps both factories in tasks, then `asyncio.wait(timeout=30.0, return_when=FIRST_COMPLETED)` waits for the first to finish.
- **`for t in pending: t.cancel()`** - stops the slower stream (you still pay for tokens already produced).

**In one line:** launch both, keep whoever speaks first, cancel the rest.

**What you'll see:** Prints the winning provider name and its first token of text, e.g. `gemini Streaming delivers tokens incrementally...`. Which side wins varies by network and load from run to run.

## 7 - Real-time speech-to-speech sketch

A minimal look at the next frontier: speech-to-speech over a single WebSocket with no STT -> LLM -> TTS chain in between. This is a transport sketch, not a full voice agent - turn-taking and interruption handling come in Lesson 17.3.

> **Needs an OpenAI Realtime key** (OPENAI_API_KEY) and a `play()` audio sink; shown mainly for reference.

In [ ]:
# Minimal OpenAI Realtime API sketch - speech-to-speech over one WebSocket
import asyncio, json, os
import websockets   # pip install websockets

URL = "wss://api.openai.com/v1/realtime?model=gpt-realtime"
HEADERS = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}

async def main():
    try:
        async with websockets.connect(
                URL, additional_headers=HEADERS, open_timeout=30) as ws:
            await ws.send(json.dumps({
                "type": "session.update",
                "session": {"modalities": ["audio", "text"],
                            "voice": "alloy"},
            }))
            async for raw in ws:
                event = json.loads(raw)
                if event["type"] == "response.audio.delta":
                    play(event["delta"])   # base64 PCM chunk out to the speaker
    except Exception as e:
        print(f"realtime session failed: {e}")

asyncio.run(main())
# Output: session.updated -> response.audio.delta frames stream back as speech.

**Explanation**

A bare `websockets` client that opens the OpenAI Realtime endpoint, configures the session for audio + text, and streams audio deltas back out to a speaker. The idea is that real-time audio is just the streaming transport you already learned, carrying base64 PCM frames instead of text tokens.

**How the code works - step by step**
- **`websockets.connect(URL, additional_headers=HEADERS, open_timeout=30)`** - opens the WebSocket to `wss://api.openai.com/v1/realtime` with the bearer token.
- **`session.update`** - configures the session for `modalities: ['audio','text']` and a voice (`alloy`).
- **`async for raw in ws`** - the receive loop; each message is a JSON event.
- **`response.audio.delta` handling** - passes the base64 PCM chunk to `play()` to emit speech as it arrives.
- **the `except`** - prints a failure message instead of dying silently mid-session.

**In one line:** open one socket, set audio mode, and play each audio delta as it streams back.

**What you'll see:** With a live key and an audio sink, the session emits `session.updated` and then a stream of `response.audio.delta` frames that play back as speech; without them it just prints a `realtime session failed: ...` line.

You built the streaming belt at every layer it can break: the async SDK loop that stamps TTFT, the SSE endpoint that names its frames and stops billing when the tab closes, the browser reader that splits on the blank line, the WebSocket handler that keeps one ear open so "stop" is heard mid-answer, and the Cloud Run flags that keep a proxy from buffering it all back into a parcel. The throughline: streaming is an agreement across your generator, framework, every proxy, and the browser - any one hop can turn the conveyor belt back into an Amazon box. Next, Lesson 7.3 (Cost Engineering) picks up the billing blind spot this lesson flagged - metering cancelled streams - and Lesson 7.4 grows the SSE skeleton here into a full chat application.